In [190]:
import torch
import pickle
import argparse
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
import re
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from terrarium.models import get_model, load_model
from terrarium.dataloaders import get_dataloader

In [191]:
model_name = 'gpt2_v1'
ckpt_path = 'out/gpt2_v1/7M/ckpt.pt'
data_dir = 'data/v_32768'
dl_name = 'standard_inference'
batch_size = 10
max_seq_len = 20
num_samples = 20
seed_lengths = [1,2,3,4,5]
targets = [261,262,263,264,265,266]
total_seqs = 100
data_prefix = 'val'
summary_only = False
out_dir = './'
device = 'cpu'

In [192]:
model = load_model(model_name,ckpt_path,device_override='cpu').to(device)
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False) # always first to cpu

In [193]:
with open(f"{data_dir}/tok.pkl",'rb') as f:
    tok = pickle.load(f)
sos_id = tok.encode('<|sos|>')
dl = get_dataloader(dl_name,batch_size,max_seq_len,f"{data_dir}/{data_prefix}_token_ids.bin",sos_id,device)

In [194]:
# check if dataset has enough sequences
if len(dl.split_locs)-1 < total_seqs:
    total_sequences = len(dl.split_locs)-1
    print(f'Warning: requested total_sequences reduced to {total_sequences} due to data size.')
n_batches = total_seqs // batch_size

In [195]:
# create the holder for results
targets = torch.tensor(targets, device=x.device, dtype=torch.long)
K = targets.numel()
results = {
    s: {
        "mc": [
            {
                "true_token": [],
                "gen_token": []
            }
            for _ in range(num_samples)
        ],
        "counts": [],
        "sampled_token": [],
        "top1_token": [],
        "cross_entropy_loss_token": []

    }
    for s in seed_lengths
}

In [ ]:
x,y = dl.next_batch()
x

In [ ]:
# assemble x and y from the return sequences t.
x_pad = torch.zeros((len(x), max_seq_len), dtype=torch.long)
y_pad = torch.zeros((len(y), max_seq_len), dtype=torch.long)
seq_lengths = torch.zeros((len(x),), dtype=torch.long)
for i, (xi, yi) in enumerate(zip(x, y)):
    L = len(xi)
    seq_lengths[i] = L
    x_pad[i, :L] = xi
    y_pad[i, :L] = yi
x_pad

In [117]:
with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.bfloat16):
    logits,_ = model.forward(x_pad,targets=y_pad) # (B,T,V)

logits.shape

torch.Size([10, 20, 32786])

In [118]:
B,T,V = logits.shape

In [119]:
rows_to_use = torch.arange(B, device=device)
rows_to_use

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [120]:
token_probs = torch.softmax(logits,dim=-1)
token_probs.shape

torch.Size([10, 20, 32786])

In [121]:
token_probs.view(-1,V)

tensor([[4.4793e-11, 5.4115e-11, 5.0932e-11,  ..., 3.8426e-11, 4.6384e-11,
         5.2523e-11],
        [8.1601e-15, 1.9429e-15, 3.6360e-15,  ..., 6.6960e-16, 5.6066e-15,
         8.7153e-15],
        [3.9290e-10, 4.3110e-10, 4.4565e-10,  ..., 4.0563e-10, 4.7294e-10,
         3.2560e-10],
        ...,
        [3.0377e-10, 6.8394e-10, 1.7280e-10,  ..., 3.2378e-10, 2.8558e-10,
         5.0204e-10],
        [4.0563e-10, 9.7498e-10, 2.3101e-10,  ..., 4.3110e-10, 3.8017e-10,
         7.1304e-10],
        [4.0745e-10, 9.1677e-10, 2.1737e-10,  ..., 4.0745e-10, 4.0745e-10,
         6.6939e-10]], dtype=torch.bfloat16)

In [122]:
torch.multinomial(token_probs.view(-1,V),num_samples=1)

tensor([[  271],
        [  258],
        [  912],
        [  399],
        [ 1700],
        [  259],
        [  934],
        [ 1864],
        [ 1170],
        [  338],
        [  787],
        [  711],
        [ 1709],
        [   47],
        [  480],
        [  266],
        [  731],
        [ 1526],
        [  816],
        [ 8276],
        [  277],
        [  258],
        [  905],
        [ 2370],
        [ 1226],
        [ 3531],
        [  622],
        [   51],
        [  100],
        [12727],
        [  388],
        [ 1065],
        [13256],
        [  920],
        [   47],
        [  260],
        [  264],
        [10032],
        [12096],
        [10032],
        [  293],
        [  258],
        [ 1891],
        [  657],
        [  259],
        [ 1618],
        [  347],
        [ 6678],
        [  388],
        [  480],
        [  597],
        [  388],
        [ 2449],
        [ 2891],
        [ 4496],
        [25773],
        [ 2157],
        [  348],
        [  399

In [123]:
sampled_token = torch.multinomial(token_probs.view(-1,V),num_samples=1).view(B,T) # B,T
# sample over V dimension, sampled_token should look similar to y_pad
print(x_pad)
print(y_pad)
sampled_token

tensor([[  257,   298,   258,  2006,   622,  3020,   259,   934,  7974,   566,
           798, 13168,  2748,  7317,  7681,   260,     0,     0,     0,     0],
        [  257,   279,   258,  3897, 12483,    33,    51,    48,    52,   259,
          3897,   388, 12483,   513,   920,    47,   260,     0,     0,     0],
        [  257,   279,   258,   755,   657,   259,   577,   900,   347,   470,
           358,  2635,  2367,   349,   578,  2935,  8389,  2157,    45,   260],
        [  257,   281,   258,   905,   338,  1241,   259,   905,   338,  1241,
           260,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  257,   295,   258,   755,   657,   259,  1087,   260,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  257,   292,   258,   755,   657,   259,  1404,   604,   324,  1098,
           260,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  257,   285,   258,  1891,  2616,   

tensor([[  300,   258, 16519,  6092,  1630,   259, 25000,  8131,   566,  2337,
          7974,   328,   391,    34,   328,   266, 26706,   343,  3496,  1526],
        [  285,   258,    81,  1226,    34,   514,   973,    53,   427,   761,
          1226,    33,   386,   920,   259,  4839,   265,  1339,  1073, 17792],
        [  271,   258,   755,   657,   259,  2348,   347,   400,   481,   328,
          7334,  1994, 12013,   388, 12040,  6944,  2157,    47,   784,   266],
        [  289,   258,  1274,   371,    34,   259,   501,   310,    47,   442,
           266,   343,   816,   380,   343,   843,  2115,   414,  2054,   343],
        [  298,   258,   536,   657,   259,  5196,  1878,   266,   731,  1382,
          1991,   123, 22992,   486,   361,  1526,  1483,  2277,   343,   459],
        [  283,   258,   536,   657,   259,  4583,   411,   400,  1188,   260,
           266,  1526,  3630,  1526, 13655,   388,   789,  9694, 25548,  1073],
        [  284,   258,   536,  2616,   259,  1

In [124]:
torch.argmax(token_probs,dim=-1) # B,T

tensor([[  277,   258,   755,   328,  2099,   259,   934,  1429,   566,   338,
          3020,   328,  7317,    47,    47,   266,  1526,  1526,  1526,  1526],
        [  277,   258,   755,  1226,   259,    51,   259,    53,   840,    74,
          1226,    33,  1226,  1058,   493,   260,   266,  1731,  1731,  1526],
        [  277,   258,   755,   657,   259,   536,   347,   347,   304,    47,
           310,   388,   349,   388,  1065,  6000,  2157,    47,   399,   266],
        [  277,   258,   755,   371,   259,   259,    74,   338,    34,   260,
           266,   325,   325,   325,   325,   789,   789,   343,   789,   789],
        [  277,   258,   755,   657,   259,   536,   260,   266,  1526,  1526,
          1526,  1526,  1526,  1526,  1526,  1526,  1526,  1526,  1526,  1526],
        [  277,   258,   755,   657,   259,   536,   411,   324,  1098,   260,
           266,  1526,  1526,  1526,  1526,  1526,  1526,  1526,  1526,  1526],
        [  277,   258,   755,  2616,   259,   

In [125]:
y_pad.shape

torch.Size([10, 20])

In [24]:
y_pad.unsqueeze(-1).shape

torch.Size([2, 20, 1])

In [25]:
token_probs.shape

torch.Size([2, 20, 32786])

In [26]:
torch.gather(token_probs,2,y_pad.unsqueeze(-1)).clamp_min(1e-9).shape

torch.Size([2, 20, 1])

In [27]:
-torch.log(torch.gather(token_probs,2,y_pad.unsqueeze(-1)).clamp_min(1e-9)).squeeze(-1) # B,T

tensor([[2.5156e+00, -0.0000e+00, 2.3594e+00, 5.2188e+00, 5.2812e+00, 2.8125e+00,
         1.2891e-01, 3.7188e+00, 1.3594e+00, 3.9375e+00, 2.4531e+00, 6.2500e+00,
         3.9062e+00, 4.8750e+00, 6.2500e+00, 3.6406e+00, 9.2578e-01, 1.9775e-02,
         2.0750e+01, 2.0750e+01],
        [2.5156e+00, -0.0000e+00, 2.3594e+00, 5.2188e+00, 5.2812e+00, 2.8125e+00,
         1.2891e-01, 3.7188e+00, 1.3594e+00, 3.9375e+00, 2.4531e+00, 6.2500e+00,
         3.9062e+00, 4.8750e+00, 6.2500e+00, 3.6406e+00, 9.2578e-01, 1.9775e-02,
         2.0750e+01, 2.0750e+01]], dtype=torch.bfloat16)

In [28]:
seq_lengths.shape

torch.Size([2])

In [29]:
len(y[0])

18

In [30]:
len(y_pad[0])

20

In [31]:
(seq_lengths - 1).clamp_min(0)  # B, # original sequence lengths before padding

tensor([17, 17])

In [32]:
x,y = dl.mc_next_batch(B_override=num_samples)

In [33]:
x

[tensor([ 257,  279,  258,  536,  622, 2891,   34,  259,  905,  515,   45,  470,
          360,  304, 8853, 8001,   34,  260]),
 tensor([ 257,  279,  258,  536,  622, 2891,   34,  259,  905,  515,   45,  470,
          360,  304, 8853, 8001,   34,  260])]

In [34]:
y

[tensor([ 279,  258,  536,  622, 2891,   34,  259,  905,  515,   45,  470,  360,
          304, 8853, 8001,   34,  260,  266]),
 tensor([ 279,  258,  536,  622, 2891,   34,  259,  905,  515,   45,  470,  360,
          304, 8853, 8001,   34,  260,  266])]

In [165]:
search_id = torch.tensor(tok.encode('<|sor|>'))

In [166]:
search_id

tensor([260])

In [197]:
# --- run the inference over batches --- #
for b in tqdm(range(total_seqs)):
    x,y = dl.mc_next_batch(B_override=num_samples)
    x,y = torch.stack(x, dim=0), torch.stack(y, dim=0)
    seq_lengths = torch.full((x.size(0),), x.size(1), device=x.device, dtype=torch.long)

    B,T = x.shape
    true_token = y[:, -1]

    for s in seed_lengths:
        if s-1 <= 0 or s-1 >= T:
            continue
        x_s = x[:,:s-1]
        y_s = y[:,:s-1]

        output = model._generate(x_s,max_seq_len-x_s.shape[1]) 
        T_out = output.size(1)

        # Find first occurrence of any token in search_id
        mask = torch.isin(output, search_id)
        found = mask.any(dim=1)
        first_occ = mask.float().argmax(dim=1)
        first_occ = first_occ + 1
        valid = found & (first_occ < T_out)
        gather_pos = first_occ.clamp(min=0, max=T_out-1)
        gen_token = output.gather(1, gather_pos.unsqueeze(1)).squeeze(1)
        gen_token[~valid] = -1 

        for n in range(x_s.size(0)):
            results[s]["mc"][n]["true_token"].append(true_token.tolist()[n])
            results[s]["mc"][n]["gen_token"].append(gen_token.tolist()[n])
    
        # perform bin counts
        valid_tokens = gen_token[valid]
        if valid_tokens.numel() > 0:
            counts = (valid_tokens.unsqueeze(1) == targets.unsqueeze(0)).sum(dim=0)  # (K,)
        else:
            counts = torch.zeros(K, device=x.device, dtype=torch.long)

        total = counts.sum()
        if total > 0:
            token_probs = counts.float() / total
        sampled_token = int(targets[torch.multinomial(token_probs, num_samples=1).item()].item())
        top1_token = int(targets[int(token_probs.argmax().item())].item())

        true_token_s = (true_token.unsqueeze(1) == targets.unsqueeze(0)).float().argmax(dim=1)   
        nll_token = -torch.log(torch.gather(token_probs.unsqueeze(0).expand(true_token.size(0), -1), 1, true_token_s.unsqueeze(1)).clamp_min(1e-9)).squeeze(1)

        results[s]["counts"].append(counts.detach().cpu().tolist())
        results[s]["sampled_token"].append(sampled_token)
        results[s]["top1_token"].append(top1_token)
        results[s]["cross_entropy_loss_token"].append(nll_token)

100%|██████████| 100/100 [03:55<00:00,  2.36s/it]


In [198]:
results

{1: {'mc': [{'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []},
   {'true_token': [], 'gen_token': []}],
  'counts': [],
  'sampled_token': [],
  'top1_token': [],
  'cross_entropy_loss_token': []},
 2: {'mc': [{'true_token': [266,
     266,
     266,
     266,
     266,
     266,
     266,
   

In [205]:
# ---- data post processing ---- #
flat = {
    "seed_len": [],
    "sample": [],
    "sampled_token": [],
    "top1_token": [],
    "cross_entropy_loss_token": []
}

# If you want counts as columns too:
targets_list = targets.detach().cpu().tolist()
for tok in targets_list:
    flat[f"count_{tok}"] = []

for s, d in results.items():
    nb = len(d["top1_token"])
    for i in range(nb):
        flat["seed_len"].append(s)
        flat["sample"].append(i)
        flat["sampled_token"].append(d["sampled_token"][i])
        flat["top1_token"].append(d["top1_token"][i])

        nll_list = d["cross_entropy_loss_token"][i]
        flat["cross_entropy_loss_token"].append(float(np.nanmean(nll_list)))

        counts = d["counts"][i]
        for j, tok in enumerate(targets_list):
            flat[f"count_{tok}"].append(counts[j])

In [206]:
df_stratified_summary = pd.DataFrame(flat)
df_stratified_summary

,seed_len,sample,sampled_token,top1_token,cross_entropy_loss_token,count_261,count_262,count_263,count_264,count_265,count_266
0,2,0,266,266,0.182322,0,0,0,0,1,5
1,2,1,266,266,0.154151,0,0,0,0,1,6
2,2,2,266,266,0.000000,0,0,0,0,0,6
3,2,3,266,266,0.405465,0,1,0,0,2,6
4,2,4,266,266,0.000000,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...
401,5,96,266,266,0.336472,0,0,0,1,1,5
402,5,97,266,266,0.000000,0,0,0,0,0,7
403,5,98,266,266,0.000000,0,0,0,0,0,18
404,5,99,266,266,0.000000,0,0,0,0,0,10


In [207]:
rows = []
for s, d in results.items():
    for n, rec in enumerate(d["mc"]):
        L = len(rec["gen_token"])
        for t in range(L):
            rows.append({
                "seed_len": s,
                "sample_id": n,
                "sample": t,
                "true_token": rec["true_token"][t],
                "gen_token": rec["gen_token"][t],
            })

final_df = pd.DataFrame(rows)

In [208]:
final_df

,seed_len,sample_id,sample,true_token,gen_token
0,2,0,0,266,-1
1,2,0,1,266,266
2,2,0,2,266,266
3,2,0,3,266,-1
4,2,0,4,266,-1
...,...,...,...,...,...
8115,5,19,96,266,-1
8116,5,19,97,266,-1
8117,5,19,98,266,266
8118,5,19,99,266,-1


In [209]:
# concatenate all dataframes
# final_df.to_csv('./out/online_classification_results.csv', index=False) # not sure want to save, could be massive
if not summary_only:
    final_df.to_csv(os.path.join(out_dir,'online_classification_results.csv'), index=False)

In [58]:
df_stratified_summary.to_csv(os.path.join(out_dir,'online_classification_stratified_summary.csv'),index=False)